# Link a PDF score to an MEI file, step by step

**Workflow 1 — editing music with the MEI data format. Guided version for beginners.** This notebook connects a scanned score (a PDF) with its MEI encoding, bar by bar. Afterwards you can point at a bar in the encoded score and see the same bar highlighted in the scan.

| Step | What the computer does | What you do |
| --- | --- | --- |
| 1. PDF → page images | turns every PDF page into a picture | look at the pictures |
| 2. Describe the pages | checks your description against the MEI | write one line per page with music |
| 3. Find the bars | a detector draws a box around every bar | check the boxes, add one the detector missed |
| 4. Merge with the MEI | writes the boxes into a copy of your MEI file | run the cell |
| 5. Look at the result | shows score and scan side by side | point and click |

The example is Dieterich Buxtehude's Suite from *Denkmäler deutscher Tonkunst*, vol. 11 (Leipzig 1903), pp. 160–163: 89 bars on four pages. The file `test_corpus/Buxtehude-DdT11-app01-suite.mei` holds its encoding. Your original MEI file is never changed; everything new goes into the folder `converted_mei/pdf_guided/`.

For several movements in one file, schema checks and more control, use the full version: [`mei_pdf_facsimile_integration.ipynb`](mei_pdf_facsimile_integration.ipynb).

## Before you start: how Jupyter works

- A notebook is made of **cells**. Grey cells contain Python code; the others contain text like this one.
- To **run** a cell, click into it and press **Shift + Enter** (or click ▶ in the toolbar). While it runs, `[*]` appears next to it; when it has finished, a number appears.
- Run the cells **from top to bottom**. Every step needs the steps before it.
- Change only the cells marked **✏️**, and in them only the text between quotes `"…"`, the numbers, and `True` / `False`. Keep the quotes, commas and brackets.
- Lines that start with `#` are comments. Python ignores them; they explain the line below.
- When something is missing, the cell prints **⚠ NOT DONE** and says what to do. To start again from a clean state, use **Kernel → Restart Kernel and Run All Cells** (JupyterLab; other programs have a similar menu entry). Your files stay where they are.

## Setup

Run the next cell. It loads CAMAT and the functions for this notebook.

In [ ]:
# Cloud Jupyter: install CAMAT and copy notebooks/test_corpus if they are not already here.
try:
    import setup_camat
except ModuleNotFoundError:
    pass
try:
    from camat.notebook_workspace import prepare_notebook
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "camat"])
    from camat.notebook_workspace import prepare_notebook

prepare_notebook()

# You can leave this cell unchanged.
import setup_camat

try:
    from camat.pdf_facsimile_guided import (
        make_page_images, describe_pages, find_bars, check_bars, merge_with_mei, open_result,
    )
except ImportError as problem:
    raise ImportError(
        "This notebook needs a CAMAT version with PDF support (camat.pdf_facsimile_guided). "
        "Update CAMAT with: python -m pip install -U camat"
    ) from problem
print("✓ CAMAT is ready.")

## ✏️ Your settings

The next cell holds all settings. For the example:

1. Write your name between the quotes of `YOUR_NAME`.
2. Set `DOWNLOAD_EXAMPLE_PDF` and `SEND_PAGES_TO_DETECTOR` to `True` once you have read what they do (steps 1 and 3 explain it).
3. Run the cell.

**Run this cell again after every change.** The steps below use what this cell last stored, not what you see in it.

In [ ]:
# ✏️ Your settings. Change the text between quotes and True / False only.

# The MEI file (the encoded music). The example file comes with CAMAT.
MEI_FILE = "test_corpus/Buxtehude-DdT11-app01-suite.mei"

# The PDF (the scanned score). Step 1 explains how to get the example PDF.
PDF_FILE = "converted_mei/pdf_guided/bsb00023199_0170-0173.pdf"

# Your name: stored in the result as the person who checked the bar boxes.
YOUR_NAME = ""

# True = yes, False = no.
DOWNLOAD_EXAMPLE_PDF = False      # download the example PDF from the Bayerische Staatsbibliothek
SEND_PAGES_TO_DETECTOR = False    # upload the music pages to the measure detector (needs internet)
MATCH_LINE_BREAKS_TO_PDF = True   # page and line breaks of the result follow the PDF

# Where the results go. You can leave this as it is.
OUTPUT_FOLDER = "converted_mei/pdf_guided"

## Step 1 — Turn the PDF into page images

The computer cannot measure bars inside a PDF directly, so CAMAT first turns every PDF page into a picture (PNG, 300 dots per inch). Each picture keeps the number of its PDF page: the first page of the file is **PDF page 1**, even when the print says "160" on it.

**Get the example PDF** in one of two ways. The scans belong to the Bayerische Staatsbibliothek (BSB) in Munich, under the licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): non-commercial use, name the source, share changed versions under the same licence.

- **Automatically:** set `DOWNLOAD_EXAMPLE_PDF = True` in the settings cell. This answers "yes" for you to the BSB's question whether you have read its rights notice and agree to it.
- **By hand:** open the [BSB download form for images 170–173](https://download.digitale-sammlungen.de/BOOKS/download.pl?id=00023199&ersteseite=170&letzteseite=173). At *"Ich versichere, den Rechtehinweis gelesen zu haben und bin damit einverstanden"* ("I confirm that I have read the rights notice and agree to it") choose **Ja**, click the **WEITER** button next to *PDF*, and save the PDF linked on the next page. Put it into the folder `converted_mei/pdf_guided/` (create the folder if needed) under the name `bsb00023199_0170-0173.pdf`, or set `PDF_FILE` to where you saved it.

The PDF has **5 pages**: page 1 is a cover sheet from the library; pages 2–5 are the printed pages 160–163. Keep the file: a new download of the same pages counts as a different file.

This step needs the free program **Poppler**. If it is missing, the cell says how to install it.

In [ ]:
make_page_images(PDF_FILE, OUTPUT_FOLDER, download_example=DOWNLOAD_EXAMPLE_PDF)

## Step 2 — ✏️ Describe the music pages

Look at the page pictures from step 1. For every page **with music**, write one line in the cell below:

```
PDF page | printed page | bars on each system, top to bottom
```

- A **system** is one line of music: all staves that sound together, joined by a brace or bracket on the left. Page 160 of the example has three systems.
- Write the bars of a system as a range like `1-4`, and separate the systems with commas.
- Use the **bar numbers of the MEI file** (`<measure n="…">`). This 1903 print has no bar numbers, so count. An upbeat counts as a bar when the MEI encodes it as its own `<measure>`: in the example, bar 10 is the short upbeat after the repeat sign at the start of the third system on page 160.
- A bar that starts at the end of one system and continues on the next gets its number twice: `10-14, 14-18`.
- Pages you do not list count as pages without music. You may name them, for example `1 | cover` (other words: title, preface, blank, advertisement, duplicate).
- Lines that start with `#` are ignored.

The example's description is already filled in. The cell compares it with the MEI file and tells you if bars are missing, extra, or in a different order.

In [ ]:
# ✏️ One line per page:  PDF page | printed page | bars on each system, top to bottom
MUSIC_PAGES = """
1 | cover
2 | 160 | 1-4, 5-9, 10-14
3 | 161 | 15-18, 19-25, 26-33, 34-40
4 | 162 | 41-47, 48-55, 56-63, 64-70
5 | 163 | 71-74, 75-79, 80-84, 85-89
"""

describe_pages(MUSIC_PAGES, MEI_FILE, OUTPUT_FOLDER)

## Step 3 — Find the bars on the pages

In MEI, a **zone** is a rectangle on a page picture. Four numbers describe it, counted in pixels from the top-left corner of the picture: **left**, **top**, **right**, **bottom**. In step 4, every bar of the MEI will point to its zone.

Drawing 89 rectangles by hand takes long, so a **measure detector** does it: a neural network of the Edirom project, at `measure-detector.edirom.de`. It looks only at the picture, not at the MEI, and returns one box per bar it recognises. It is good but not perfect, so you check its work in step 3b.

### 3a — Let the detector find the bars

With `SEND_PAGES_TO_DETECTOR = True`, the pictures of the music pages are uploaded to the detector. Its answers are saved in your output folder, so running this cell again uploads nothing. Every box gets a small number; you need these numbers only to remove a wrong box.

In [ ]:
find_bars(OUTPUT_FOLDER, send_to_detector=SEND_PAGES_TO_DETECTOR)

### 3b — ✏️ Check the boxes and add the ones the detector missed

The cell below compares the boxes with your description from step 2, system by system.

- **If they match,** it labels every box with its bar number (movement/bar, for example `1/10`) and shows the pages. Check that every label sits on the right bar.
- **If they do not match,** it says which system is wrong and shows that system enlarged, with rulers in pixels.

**Exercise: add a missing box yourself.** The detector misses **bar 10** on PDF page 2, the short upbeat after the repeat sign at the start of the third system. Run the cell once as it is and look at the enlarged picture. Then work out the four numbers of the missing box:

1. **top** and **bottom**: copy them from the neighbouring box in the list under the picture. The new box covers the same staves.
2. **left**: where the staff lines begin. Read it on the ruler.
3. **right**: just after the first barline. The box may overlap the next box a little; the detector's boxes overlap too.

Write them into `ADD_BOXES` in the order (left, top, right, bottom): remove the `#` in front of the prepared line, put in your numbers, and run the cell again. Small differences do not matter.

<details>
<summary>Solution (open it only after trying)</summary>

```python
ADD_BOXES = {
    2: [(180, 2540, 530, 3560)],
}
```

Anything from about 150 to 250 for *left* and 500 to 560 for *right* works.
</details>

A box around something that is not a bar (a title, say) goes into `REMOVE_BOXES`, with the number shown in the pictures of step 3a: `{3: [7]}` removes box 7 on PDF page 3. To undo a correction, delete it and run the cell again.

In [ ]:
# ✏️ Boxes to add:     {PDF page: [(left, top, right, bottom), ...]}
ADD_BOXES = {
    # 2: [(left, top, right, bottom)],
}

# ✏️ Boxes to remove:  {PDF page: [box number, ...]}
REMOVE_BOXES = {
}

check_bars(OUTPUT_FOLDER, ADD_BOXES, REMOVE_BOXES)

## Step 4 — Merge the boxes into the MEI

This step writes a **new** MEI file; your original stays as it is. CAMAT adds a `<facsimile>` section with one `<surface>` per music page. A surface holds a `<graphic>` (the page picture) and one `<zone>` per box. Then every `<measure>` gets a `facs` attribute that points to its zone:

```xml
<surface xml:id="surface-…-p0002" n="160" …>
   <graphic target="../pages/page-0002.png" …/>
   <zone xml:id="zone-…-p0002-001" type="measure" ulx="554" uly="601" lrx="944" lry="1578" …/>
   …
<measure n="1" facs="#zone-…-p0002-001" …>
```

- `YOUR_NAME` is stored as the person who checked the boxes in step 3b. If it is empty, the cell reminds you.
- With `MATCH_LINE_BREAKS_TO_PDF = True`, the page breaks (`<pb>`) and system breaks (`<sb>`) of the new file follow the PDF, so the score in step 5 breaks its lines where the print does.
- The new file finds the page pictures by a relative path (`../pages/…`). Keep the output folder together when you copy it.
- Running this step again replaces the new file. If you edit the new file by hand, copy it somewhere else first.

In [ ]:
merge_with_mei(MEI_FILE, OUTPUT_FOLDER, checked_by=YOUR_NAME, match_line_breaks=MATCH_LINE_BREAKS_TO_PDF)

## Step 5 — Look at the result

The viewer shows the encoded score (drawn by Verovio) next to the page pictures. Move the mouse over a bar on either side and its partner lights up. Use the toolbar to turn pages and zoom. The viewer only reads the file.

To open the result again later, use [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb) and set `MEI_SOURCE` to the new file printed in step 4.

In [ ]:
viewer = open_result(MEI_FILE, OUTPUT_FOLDER)

## Your own score

1. In the settings cell, set `MEI_FILE` to your MEI file (one movement) and `PDF_FILE` to your PDF, for example `"notebooks/my_score.pdf"` if you put the PDF next to this notebook. Set `DOWNLOAD_EXAMPLE_PDF = False`.
2. Give every score its own `OUTPUT_FOLDER`, for example `"converted_mei/pdf_guided_my_score"`. Run the settings cell.
3. Run step 1 and look at the pictures.
4. Replace the example lines in step 2 with your pages, and run step 2 until it shows ✓.
5. Run step 3a. In step 3b, empty `ADD_BOXES` and `REMOVE_BOXES` first, then fix what the cell reports.
6. Run steps 4 and 5.

## If something goes wrong

| What you see | What to do |
| --- | --- |
| `NameError: name '…' is not defined` | Run the cells from the top: setup, settings, then the steps in order. |
| You changed a setting, but nothing changed | Run the settings cell again after every change. |
| `SyntaxError` in a ✏️ cell | A quote, comma or bracket is missing. Compare with the example lines. |
| Step 1: "needs the free program Poppler" | Install it as the message says, restart the kernel, run again from the top. |
| Step 1: "already holds pictures of another PDF" | You changed the PDF (a new download also counts). Choose a new `OUTPUT_FOLDER`. |
| Step 2: bars missing or extra | Compare your lines with the print and with the bar numbers in the MEI. |
| Step 3b: a box is missing or too many | Do the exercise in step 3b with your own numbers. |
| Step 3b: a label sits on the wrong bar | Your description in step 2 does not match the print. Correct it, then run steps 2, 3a and 3b again. |